# §4.3–4.4 — Safety Interventions: Statistical Tests

Tests whether compliance prompts (weak, strong, full) reduce ISR,
under both low-pressure (Tiers 1–3) and high-pressure (Tiers 4–7).
All tests use model-level means (n=16).

**§4.3 — Low pressure (Tiers 1–3):**
- 3 safety × 3 tiers = 9 Wilcoxon tests + Holm correction
- Dose-response: 3 pairwise (weak vs strong, strong vs full, weak vs full)

**§4.4 — High pressure (Tiers 4–7):**
- 3 safety × 4 tiers = 12 Wilcoxon tests + Holm correction
- Capability-gating: Spearman ρ(baseline ISR, δ) where δ = ISR(strong) − ISR(full)

**Outputs:**
- `latex/tables/safety_low_stats.tex`
- `latex/tables/safety_high_stats.tex`

In [1]:
import sys, json
import os
from pathlib import Path
import numpy as np
from scipy import stats as sp
from statsmodels.stats.multitest import multipletests

ROOT = Path(os.environ.get('HIREWATCH_ROOT', '.')).resolve()
RESULTS = ROOT / 'results' / 'interview_v4'
TABLES_DIR = ROOT / 'outputs' / 'tables'

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from analysis.labels import (
    MODELS, MODEL_SLUGS, TIER_LADDER, TIER_META,
    SAFETY_LEVELS,
)
from analysis.validation import validate_all_json_file

PERSONA = 'B'
TEMP = 0.0

# Safety levels excluding 'none'
SAFETY_ACTIVE = [(suffix, col, label) for suffix, col, label in SAFETY_LEVELS if suffix]
print(f'Active safety levels: {[l for _, _, l in SAFETY_ACTIVE]}')

Active safety levels: ['Weak', 'Strong', 'Legal text']


In [2]:
def load_model_mean_isr(slug, cond):
    """Return mean ISR for one (model, condition, temp, persona) cell."""
    p = RESULTS / cond / slug / f'temp_{TEMP}' / f'persona{PERSONA}' / 'all_results.json'
    if not p.exists():
        return None
    trials = validate_all_json_file(
        p, temperature=TEMP, persona=PERSONA, canonical_models=MODEL_SLUGS
    )
    return np.mean([t['isr'] for t in trials])

# Build ISR matrix: isr[model_idx][tier_slug][safety_suffix]
tier_slugs = [slug for _, slug, _, _ in TIER_LADDER]
tier_labels = [TIER_META[s].get('short', s) for s in tier_slugs]

isr = {}  # isr[(model_idx, tier_slug, safety_suffix)] = mean ISR
for mi, (slug, name, _) in enumerate(MODELS):
    for ts in tier_slugs:
        for suffix, _, _ in SAFETY_LEVELS:
            cond = f'{ts}{suffix}'
            val = load_model_mean_isr(slug, cond)
            isr[(mi, ts, suffix)] = val

print(f'Loaded {sum(1 for v in isr.values() if v is not None)} cells')

Loaded 504 cells


## §4.3 — Safety under low pressure (Tiers 1–3)

Δ_safe = ISR(no prompt) − ISR(safety). Positive = safety helps.

In [3]:
low_tiers = [slug for n, slug, _, _ in TIER_LADDER if n <= 3]
low_labels = [TIER_META[s]['short'] for s in low_tiers]

# 9 tests: 3 safety × 3 tiers
raw_results_low = []
for ts, tl in zip(low_tiers, low_labels):
    for suffix, col, slabel in SAFETY_ACTIVE:
        deltas = []
        for mi in range(len(MODELS)):
            v_none = isr.get((mi, ts, ''))
            v_safe = isr.get((mi, ts, suffix))
            if v_none is not None and v_safe is not None:
                deltas.append(v_none - v_safe)
        deltas = np.array(deltas)
        median_d = np.median(deltas) * 100
        if np.all(deltas == 0):
            W, p = 0, 1.0
        else:
            W, p = sp.wilcoxon(deltas, alternative='greater')
        raw_results_low.append({
            'tier': tl, 'safety': slabel, 'median': median_d,
            'W': W, 'p_raw': p, 'n': len(deltas)
        })

# Holm correction
_, p_holm_low, _, _ = multipletests([r['p_raw'] for r in raw_results_low], method='holm')

print(f'{"Tier":<12} {"Safety":<10} {"Median Δ":>10} {"W":>8} {"p_raw":>10} {"p_holm":>10} {"Sig":>5}')
print('-' * 70)
for r, ph in zip(raw_results_low, p_holm_low):
    sig = '***' if ph < 0.001 else '**' if ph < 0.01 else '*' if ph < 0.05 else 'NS'
    r['p_holm'] = ph
    r['sig'] = sig
    print(f'{r["tier"]:<12} {r["safety"]:<10} {r["median"]:>+9.2f}pp {r["W"]:>8.0f} '
          f'{r["p_raw"]:>10.4g} {ph:>10.4g} {sig:>5}')

print(f'\nHolm correction across {len(raw_results_low)} tests.')
print(f'Note: low baseline ISR limits headroom for safety effects.')

Tier         Safety       Median Δ        W      p_raw     p_holm   Sig
----------------------------------------------------------------------
Baseline     Weak           +0.04pp       65    0.08635     0.1727    NS
Baseline     Strong         +0.06pp      116  0.0006149   0.004919    **
Baseline     Legal text     +0.06pp      100    0.01053    0.04214     *
Job duties   Weak           +0.02pp       71     0.1227     0.1727    NS
Job duties   Strong         +0.18pp      147  0.0004082   0.003674    **
Job duties   Legal text     +0.14pp      105    0.00525     0.0315     *
Job context  Weak           +0.04pp       83    0.02765    0.08294    NS
Job context  Strong         +0.27pp      130  0.0006483   0.004919    **
Job context  Legal text     +0.17pp      102   0.007788    0.03894     *

Holm correction across 9 tests.
Note: low baseline ISR limits headroom for safety effects.


### Dose-response (low pressure)

Do stronger prompts produce larger ISR reductions? Pairwise
within-model comparisons: weak vs strong, strong vs full, weak vs full.

In [4]:
# Dose-response: average Δ_safe across tiers 1-3 per model, then compare levels
model_avg_delta = {}  # model_avg_delta[safety_suffix] = array of 16 means
for suffix, col, slabel in SAFETY_ACTIVE:
    vals = []
    for mi in range(len(MODELS)):
        d_list = []
        for ts in low_tiers:
            v_none = isr.get((mi, ts, ''))
            v_safe = isr.get((mi, ts, suffix))
            if v_none is not None and v_safe is not None:
                d_list.append(v_none - v_safe)
        vals.append(np.mean(d_list) if d_list else 0)
    model_avg_delta[suffix] = np.array(vals)

dose_pairs = [
    ('_safe_weak', '_safe', 'Weak vs Strong'),
    ('_safe', '_safe_cfr', 'Strong vs Full'),
    ('_safe_weak', '_safe_cfr', 'Weak vs Full'),
]
dose_raw_p = []
dose_results = []
for s1, s2, label in dose_pairs:
    diffs = model_avg_delta[s2] - model_avg_delta[s1]  # positive = s2 reduces more
    median_d = np.median(diffs) * 100
    if np.all(diffs == 0):
        W, p = 0, 1.0
    else:
        W, p = sp.wilcoxon(diffs, alternative='greater')
    dose_raw_p.append(p)
    dose_results.append({'pair': label, 'median': median_d, 'W': W, 'p_raw': p})

_, dose_p_holm, _, _ = multipletests(dose_raw_p, method='holm')

print('Dose-response (low pressure, averaged across tiers 1-3):')
print(f'{"Pair":<20} {"Median Δ":>10} {"W":>8} {"p_holm":>10} {"Sig":>5}')
print('-' * 55)
for r, ph in zip(dose_results, dose_p_holm):
    sig = '***' if ph < 0.001 else '**' if ph < 0.01 else '*' if ph < 0.05 else 'NS'
    r['p_holm'] = ph
    r['sig'] = sig
    print(f'{r["pair"]:<20} {r["median"]:>+9.2f}pp {r["W"]:>8.0f} {ph:>10.4g} {sig:>5}')

Dose-response (low pressure, averaged across tiers 1-3):
Pair                   Median Δ        W     p_holm   Sig
-------------------------------------------------------
Weak vs Strong           +0.12pp      124    0.00564    **
Strong vs Full           +0.00pp       38     0.6999    NS
Weak vs Full             +0.06pp       88    0.02379     *


## §4.4 — Safety under pressure (Tiers 4–7)

Same design but under organizational pressure.

In [5]:
high_tiers = [slug for n, slug, _, _ in TIER_LADDER if n >= 4]
high_labels = [TIER_META[s]['short'] for s in high_tiers]

# 12 tests: 3 safety × 4 tiers
raw_results_high = []
for ts, tl in zip(high_tiers, high_labels):
    for suffix, col, slabel in SAFETY_ACTIVE:
        deltas = []
        for mi in range(len(MODELS)):
            v_none = isr.get((mi, ts, ''))
            v_safe = isr.get((mi, ts, suffix))
            if v_none is not None and v_safe is not None:
                deltas.append(v_none - v_safe)
        deltas = np.array(deltas)
        median_d = np.median(deltas) * 100
        if np.all(deltas == 0):
            W, p = 0, 1.0
        else:
            W, p = sp.wilcoxon(deltas, alternative='greater')
        raw_results_high.append({
            'tier': tl, 'safety': slabel, 'median': median_d,
            'W': W, 'p_raw': p, 'n': len(deltas)
        })

_, p_holm_high, _, _ = multipletests([r['p_raw'] for r in raw_results_high], method='holm')

print(f'{"Tier":<12} {"Safety":<12} {"Median Δ":>10} {"W":>8} {"p_raw":>10} {"p_holm":>10} {"Sig":>5}')
print('-' * 72)
for r, ph in zip(raw_results_high, p_holm_high):
    sig = '***' if ph < 0.001 else '**' if ph < 0.01 else '*' if ph < 0.05 else 'NS'
    r['p_holm'] = ph
    r['sig'] = sig
    print(f'{r["tier"]:<12} {r["safety"]:<12} {r["median"]:>+9.2f}pp {r["W"]:>8.0f} '
          f'{r["p_raw"]:>10.4g} {ph:>10.4g} {sig:>5}')

print(f'\nHolm correction across {len(raw_results_high)} tests.')

Tier         Safety         Median Δ        W      p_raw     p_holm   Sig
------------------------------------------------------------------------
Memo         Weak             +0.88pp      150  0.0002509   0.001024    **
Memo         Strong           +1.19pp      153  0.0001462   0.001024    **
Memo         Legal text       +1.25pp      153  0.0001465   0.001024    **
Authority    Weak             +0.24pp      142  0.0008898   0.001024    **
Authority    Strong           +0.38pp      136  0.0002179   0.001024    **
Authority    Legal text       +0.38pp      136  0.0002189   0.001024    **
Policy       Weak             +5.72pp      171  3.815e-06  4.578e-05   ***
Policy       Strong          +10.59pp      171  3.815e-06  4.578e-05   ***
Policy       Legal text      +10.91pp      171  3.815e-06  4.578e-05   ***
Insurance    Weak             +5.61pp      166  3.815e-05  0.0003052   ***
Insurance    Strong           +7.53pp      169  1.144e-05   0.000103   ***
Insurance    Legal text     

### Capability-gating test

Does the full legal text prompt help capable models more than weaker ones?
δ = ISR(strong) − ISR(full), averaged across pressure tiers.
Positive δ = full text helped more than strong.

**Primary capability proxy:** pooled Tier 4–7 no-safety ISR, which
preserves genuine cross-model variance (std ~10pp). This addresses the
floor-effect issue with Tier 3 (std ~1.7pp, 4/18 models at exact zero).

**Secondary:** Tier 3 ISR (reported for comparison with earlier framing).

A negative ρ means pressure-sensitive models (high x-axis) show
smaller strong-vs-full difference, which would support capability-gating.

In [ ]:
# δ = ISR(strong) − ISR(full), averaged across pressure tiers (T4–7)
# Positive δ = full text reduced ISR more than strong alone
tier3_slug = [s for n, s, _, _ in TIER_LADDER if n == 3][0]

delta_cap = []       # per-model δ
tier3_isr = []       # per-model Tier 3 ISR (secondary capability proxy)
t47_nosafe_isr = []  # per-model pooled T4–7 no-safety ISR (PRIMARY proxy)
model_names_cap = []

for mi, (slug, name, _) in enumerate(MODELS):
    t3 = isr.get((mi, tier3_slug, ''))
    high_nosafe_vals = [isr.get((mi, ts, '')) for ts in high_tiers]
    high_nosafe_vals = [v for v in high_nosafe_vals if v is not None]
    t47 = np.mean(high_nosafe_vals) if high_nosafe_vals else None
    if t3 is None or t47 is None:
        continue
    tier3_isr.append(t3)
    t47_nosafe_isr.append(t47)

    # δ across pressure tiers
    d_list = []
    for ts in high_tiers:
        v_strong = isr.get((mi, ts, '_safe'))
        v_full = isr.get((mi, ts, '_safe_cfr'))
        if v_strong is not None and v_full is not None:
            d_list.append(v_strong - v_full)  # positive = full helped more
    delta_cap.append(np.mean(d_list) if d_list else 0)
    model_names_cap.append(name)

tier3_arr = np.array(tier3_isr)
t47_arr = np.array(t47_nosafe_isr)
delta_arr = np.array(delta_cap)

# Primary: T4–7 no-safety (has genuine variance)
rho_cap, p_cap = sp.spearmanr(t47_arr, delta_arr)
# Secondary: T3 (floor-limited; kept for comparison)
rho_t3, p_t3 = sp.spearmanr(tier3_arr, delta_arr)

print('Capability-gating test:')
print(f'  δ = ISR(strong) − ISR(full), averaged across tiers 4-7')
print(f'  Positive δ = full legal text reduces ISR more than strong prompt')
print(f'  Primary:   Spearman ρ(T4–7 no-safety, δ) = {rho_cap:+.3f}, p = {p_cap:.4g}')
print(f'  Secondary: Spearman ρ(T3 ISR,          δ) = {rho_t3:+.3f}, p = {p_t3:.4g}')
print()

if p_cap < 0.05:
    if rho_cap < 0:
        print('→ Significant negative: pressure-sensitive models benefit LESS from full text.')
        print('  Supports capability-gating: legal text injection is capability-dependent.')
    else:
        print('→ Significant positive: pressure-sensitive models benefit MORE from full text.')
        print('  Does NOT support capability-gating.')
else:
    print('→ Not significant: no evidence of capability-gating.')
    print('  Full legal text has similar marginal effect across the capability spectrum.')

print(f'\n{"Model":<20} {"T3 ISR%":>8} {"T4-7 ISR%":>10} {"δ (pp)":>8}')
print('-' * 52)
for name, t3, t47, d in zip(model_names_cap, tier3_arr, t47_arr, delta_arr):
    print(f'{name:<20} {t3*100:>7.2f}% {t47*100:>9.2f}% {d*100:>+7.2f}')

## LaTeX tables

In [ ]:
def sig_star(p):
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    return ''

# Map short tier name → "Tier N: short" for intuitive reference in tables.
SHORT_TO_NUM = {
    TIER_META[slug]['short']: n
    for n, slug, _, _ in TIER_LADDER
}
def tier_display(short):
    n = SHORT_TO_NUM.get(short)
    return f'T{n}: {short}' if n is not None else short

N = len(MODELS)

# ---- Table: Low pressure (§4.3) ----
lines = []
lines.append('\\begin{table}[t]')
lines.append('\\centering')
lines.append('\\small')
lines.append('\\begin{tabular}{llrrrrl}')
lines.append('\\toprule')
lines.append('\\textbf{Tier} & \\textbf{Safety} & \\textbf{$\\boldsymbol{n}$} & \\textbf{Median $\\boldsymbol{\\Delta}$} & \\textbf{$\\boldsymbol{W}$} & \\textbf{$\\boldsymbol{p}_{\\textbf{Holm}}$} & \\textbf{Sig.} \\\\')
lines.append('\\midrule')
for r in raw_results_low:
    def fmt_p(p):
        if p == 0.0:
            return '$< 10^{-16}$'
        if p < 1e-4:
            exp = int(np.floor(np.log10(p)))
            man = p / (10 ** exp)
            return f'${man:.1f}{{\\times}}10^{{{exp}}}$'
        return f'${p:.4f}$'
    lines.append(f'{tier_display(r["tier"])} & {r["safety"]} & {r["n"]} & '
                 f'{r["median"]:+.2f}\\,pp & {r["W"]:.0f} & '
                 f'{fmt_p(r["p_holm"])} & {r["sig"]} \\\\')
lines.append('\\bottomrule')
lines.append('\\end{tabular}')
lines.append('\\caption{Safety intervention effects under low pressure '
             f'(Tiers 1--3, $T{{=}}0.0$, $n{{=}}{N}$). '
             '$\\Delta_{\\text{safe}} = \\text{ISR}(\\text{none}) - '
             '\\text{ISR}(\\text{safety})$; positive = safety helps. '
             'Holm--Bonferroni correction across 9 tests.}')
lines.append('\\label{tab:safety-low-stats}')
lines.append('\\end{table}')
tex_low = '\n'.join(lines) + '\n'
(TABLES_DIR / 'safety_low_stats.tex').write_text(tex_low)
print(f'Wrote safety_low_stats.tex')
print(tex_low)

# ---- Table: High pressure (§4.4) — side-by-side layout ----
# Split: left = Memo + Authority (6 rows), right = Policy + Insurance (6 rows)
left_tiers = ['Memo', 'Authority']
right_tiers = ['Policy', 'Insurance']
left_rows = [r for r in raw_results_high if r['tier'] in left_tiers]
right_rows = [r for r in raw_results_high if r['tier'] in right_tiers]

lines = []
lines.append('\\begin{table}[t]')
lines.append('\\centering')
lines.append('\\small')
lines.append('\\setlength{\\tabcolsep}{4pt}')
lines.append('\\begin{tabular}{llrl|llrl}')
lines.append('\\toprule')
lines.append('\\textbf{Tier} & \\textbf{Safety} & \\textbf{Median $\\boldsymbol{\\Delta}$} & \\textbf{$\\boldsymbol{W}$} &'
             ' \\textbf{Tier} & \\textbf{Safety} & \\textbf{Median $\\boldsymbol{\\Delta}$} & \\textbf{$\\boldsymbol{W}$} \\\\')
lines.append('\\midrule')
for lr, rr in zip(left_rows, right_rows):
    l_sig = sig_star(lr['p_holm'])
    r_sig = sig_star(rr['p_holm'])
    l_med = f'{lr["median"]:+.2f}\\,pp{l_sig}'
    r_med = f'{rr["median"]:+.2f}\\,pp{r_sig}'
    lines.append(f'{tier_display(lr["tier"])} & {lr["safety"]} & {l_med} & {lr["W"]:.0f} &'
                 f' {tier_display(rr["tier"])} & {rr["safety"]} & {r_med} & {rr["W"]:.0f} \\\\')
lines.append('\\midrule')
lines.append(f'\\multicolumn{{8}}{{l}}{{Capability-gating: Spearman '
             f'$\\rho(\\text{{T4--7 no-safety}}, \\delta) = {rho_cap:+.2f}$, '
             f'$p = {p_cap:.2f}$ (NS)}} \\\\')
lines.append('\\bottomrule')
lines.append('\\end{tabular}')
lines.append('\\caption{Safety intervention effects under organizational pressure '
             f'(Tiers 4--7, $T{{=}}0.0$, $n{{=}}{N}$). '
             '$\\Delta_{\\text{safe}} = \\text{ISR}(\\text{none}) - '
             '\\text{ISR}(\\text{safety})$; positive = safety reduces ISR. '
             'Holm--Bonferroni correction across 12 tests. '
             'Significance: {*}{*}{*}\\,$p < 0.001$, {*}{*}\\,$p < 0.01$, {*}\\,$p < 0.05$. '
             'Capability-gating row: Spearman correlation between model capability '
             '(pooled Tier~4--7 no-safety ISR) and $\\delta = \\text{ISR}(\\text{strong}) '
             f'- \\text{{ISR}}(\\text{{full}})$. The Tier~3 ISR proxy '
             f'(at floor for most models) yields a similar null ($\\rho = {rho_t3:+.2f}$, '
             f'$p = {p_t3:.2f}$).}}')
lines.append('\\label{tab:safety-high-stats}')
lines.append('\\end{table}')
tex_high = '\n'.join(lines) + '\n'
(TABLES_DIR / 'safety_high_stats.tex').write_text(tex_high)
print(f'\nWrote safety_high_stats.tex')
print(tex_high)